## 01 Document Loaders

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

# Load a website
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
loader = WebBaseLoader(url)
documents = loader.load()

print(f"Loaded {len(documents)} documents.")
print(documents[0].page_content[:200])


## 02 Text Splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")


## 03 Embeddings And Vectorstores

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Initialize the embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Create the Vector Store from our chunks
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    persist_directory="./chroma_db"
)

# 3. Create a Retriever interface
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("What is Task Decomposition?")
print(f"Found {len(results)} relevant chunks.")


## 04 Advanced Retrievers

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_groq import ChatGroq

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)

# MultiQueryRetriever uses an LLM to generate multiple variations of the user's query
advanced_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever, llm=llm
)

results = advanced_retriever.invoke("How do agents break down tasks?")
print(f"Advanced retrieval found {len(results)} chunks.")
